# B01 · Python 环境与语法基础

> 阶段〇（学前基础）第 1 周。
> 本周目标：搞清楚工程环境为什么这样搭（uv + Jupyter），并把 Python 最核心的语法
> （变量、控制流、函数、列表切片）用自动化专业熟悉的例子跑一遍。
> 本阶段所有例子都来自信号、电路、弹簧、单摆、PID——为后续 RL/ROS2/Isaac 课程铺路。

## 学习目标

学完本 notebook，你应该能够：

1. 说清楚为什么本项目用 `uv` 管理依赖而不是 `pip install` 到全局环境；
2. 在 Jupyter 中运行 cell，区分代码 cell 与 markdown cell；
3. 使用变量、数值类型、字符串（f-string）与布尔运算描述物理量；
4. 用 `if/for/while` 写出简单的仿真循环（欧拉法数值积分）；
5. 定义带默认参数的函数，理解局部变量与全局变量的作用域；
6. 对列表做索引与切片，提取信号片段。

> 先修要求：零基础可学。只要会打开终端、会按 `Shift+Enter` 运行 cell。

## 1. 工程环境：uv 与 Jupyter

### 1.1 为什么不用 `pip install` 全局安装？

「装个包而已，直接 `pip install numpy` 不行吗？」——能跑，但会埋雷：

- **版本冲突**：全局环境里所有项目共用一份包。项目 A 要 `numpy 1.x`，项目 B 要 `numpy 2.x`，
  一升级就把 A 搞坏（机器人领域尤其常见：ROS、Isaac、SB3 对依赖版本都很挑剔）。
- **不可复现**：三个月后换台电脑，你根本记不清当时装了哪些包的哪个版本，
  「在我机器上能跑」成了玄学。
- **污染环境**：全局 `pip install` 甚至可能弄坏系统 Python（很多 Linux 工具依赖它）。

本项目的做法是**虚拟环境 + 锁定文件**：

| 工具 | 作用 | 类比 |
|------|------|------|
| `.venv/` | 项目专属的沙盒环境，与全局隔离 | 每个实验台独立的电源 |
| `pyproject.toml` | 声明「需要哪些包」 | 元件清单（BOM 表） |
| `uv.lock` | 锁定「每个包的确切版本」，保证可复现 | 带版本号的 BOM 表 |
| `uv` | 高速包管理器，负责安装/同步/运行 | 自动采购 + 装配 |

### 1.2 本项目常用命令

```bash
uv sync                 # 按 uv.lock 同步环境（克隆项目后第一件事）
uv add <package>        # 新增依赖（会自动更新 pyproject.toml 和 uv.lock）
uv run <command>        # 在项目环境里运行命令，如 uv run pytest
uv run jupyter lab      # 启动 Jupyter（kernel 自动选中 .venv）
```

> 约定：本项目**只用 uv 管理依赖**，不要手动 `pip install` 到 `.venv`。

### 1.3 Jupyter 的两个概念

- **cell（单元格）**：代码 cell 按 `Shift+Enter` 运行；markdown cell 用来写讲解（你现在读的就是）。
- **kernel（内核）**：背后真正执行代码的 Python 进程。cell 可以乱序执行，
  变量会留在 kernel 里——这也是新手最常见的坑（改了上面的 cell 忘了重跑）。
  **交付实验前请用「Restart Kernel and Run All Cells」从头跑一遍**。

In [1]:
# 确认当前 kernel 用的是项目 .venv 里的 Python
import sys

print("Python 版本:", sys.version)
print("解释器路径:", sys.executable)

import numpy
print("NumPy 版本:", numpy.__version__)
# sys.executable 应该指向 robot_rl_learn/.venv/... 而不是 /usr/bin/python

Python 版本: 3.11.15 (main, May  4 2026, 21:12:26) [Clang 22.1.3 ]
解释器路径: /data/wangf/robot_rl_learn/.venv/bin/python
NumPy 版本: 2.4.6


## 2. 变量与数值类型

Python 变量是「名字贴在对象上」，不需要提前声明类型（动态类型）。
自动化里最常见的两类数值：`int`（整数，如采样点数）和 `float`（浮点数，如增益、时间常数）。

In [2]:
Ts = 0.01          # 采样周期 10 ms（float，单位 s）
K = 2.5            # 比例增益（float）
n_samples = 1000   # 采样点数（int）
model_name = "first_order"  # 字符串（str）

print(type(Ts), type(n_samples), type(model_name))

# 算术运算：注意 / 与 // 的区别
print("总仿真时长:", Ts * n_samples, "s")
print(7 / 2)    # 3.5  真除法，结果总是 float
print(7 // 2)   # 3    整除
print(7 % 2)    # 1    取余
print(2 ** 10)  # 1024 幂运算（不是 ^ ！）

# 类型转换
print(int(3.99), float(5), str(2.5) + " V")

<class 'float'> <class 'int'> <class 'str'>
总仿真时长: 10.0 s
3.5
3
1
1024
3 5.0 2.5 V


**浮点数警告**：`float` 是二进制近似存储，`0.1 + 0.2 != 0.3`。
比较浮点数永远用「误差带」：`abs(a - b) < 1e-9`，不要用 `==`。这一点在数值仿真里极其重要。

In [3]:
print(0.1 + 0.2 == 0.3)            # False！
print(abs(0.1 + 0.2 - 0.3) < 1e-9) # True —— 正确姿势

False
True


## 3. 字符串与布尔

字符串处理在机器人工程里无处不在：解析传感器报文、拼接日志、生成文件名。
**f-string**（Python 3.6+）是最常用的格式化手段：在字符串前加 `f`，用 `{}` 嵌入表达式。

In [4]:
sensor = "PT100"        # 铂电阻温度传感器
temp = 23.4567          # 当前读数
threshold = 80.0        # 报警阈值

# f-string：{变量:格式}，.1f 表示保留 1 位小数
msg = f"传感器 {sensor} 当前读数：{temp:.1f} degC，阈值 {threshold:.0f} degC"
print(msg)

# 布尔值与比较运算：结果是 True / False
is_alarm = temp > threshold
print("超温报警?", is_alarm, type(is_alarm))

# 逻辑运算：and / or / not（不是 && || !）
high = temp > threshold
sensor_ok = True
print("触发停机?", high and sensor_ok)
print("需要关注?", high or not sensor_ok)

传感器 PT100 当前读数：23.5 degC，阈值 80 degC
超温报警? False <class 'bool'>
触发停机? False
需要关注? False


## 4. 条件分支：`if / elif / else`

经典例子：根据阻尼比 $\zeta$ 判断二阶系统的响应形态——这正是自动控制原理里的分类。

In [5]:
zeta = 0.3  # 阻尼比

if zeta < 0:
    desc = "负阻尼（发散，系统不稳定）"
elif zeta == 0:
    desc = "无阻尼（等幅振荡）"
elif zeta < 1:
    desc = "欠阻尼（衰减振荡，有超调）"
elif zeta == 1:
    desc = "临界阻尼（最快的无超调响应）"
else:
    desc = "过阻尼（缓慢爬升，无超调）"

print(f"zeta = {zeta} → {desc}")

# Python 的缩进就是语法：同一代码块必须对齐（约定 4 个空格）

zeta = 0.3 → 欠阻尼（衰减振荡，有超调）


## 5. 循环：`for` 与 `while`

- `for`：**已知次数/已知序列**时遍历，常配 `range()`；
- `while`：**不知道要循环几次、只知道停止条件**时使用（如「迭代到误差足够小」）。

下面用一个贯穿全学期的例子——**一阶惯性环节** $ \tau \dot{y} + y = Ku $ 的欧拉法离散仿真：

$$y_{k+1} = y_k + T_s \cdot \frac{K u - y_k}{\tau}$$

In [6]:
K, tau, Ts = 2.0, 0.5, 0.01   # 增益、时间常数、采样周期
u = 1.0                        # 阶跃输入

# for 循环：固定仿真 5 s（501 个采样点）
y = 0.0
ys = []                        # 用列表收集每个时刻的输出
for k in range(501):
    ys.append(y)
    y += Ts * (K * u - y) / tau   # 欧拉法：y_{k+1} = y_k + Ts * dy/dt

print("样本数:", len(ys))
print("稳态值（数值）:", round(ys[-1], 4), " 理论值 K =", K)

样本数: 501
稳态值（数值）: 1.9999  理论值 K = 2.0


In [7]:
# while 循环：仿真到「进入稳态误差带」为止
y, k = 0.0, 0
band = 0.02 * K                # ±2% 误差带
while abs(y - K) > band:
    y += Ts * (K * u - y) / tau
    k += 1
    if k > 10 ** 6:            # 保险丝：防止逻辑错误导致死循环
        print("超过最大迭代次数，强制退出")
        break

t_settle = k * Ts
print(f"进入 ±2% 误差带用时 ≈ {t_settle:.2f} s（理论调节时间 4τ = {4 * tau:.2f} s）")

# break 跳出循环；continue 跳过本次迭代直接进入下一轮

进入 ±2% 误差带用时 ≈ 1.94 s（理论调节时间 4τ = 2.00 s）


## 6. 函数：定义与参数

函数把「一段可复用的计算」封装起来。要点：

- `def 函数名(参数):` + 缩进函数体；`return` 返回结果（可同时返回多个值）；
- **默认参数**让常用取值不必每次手写：`def f(t, K=1.0)`；
- **docstring**（函数体第一行的字符串）说明用途，是对自己和他人的承诺。

In [8]:
import math

def first_order_step(t, K=1.0, tau=1.0):
    '''一阶惯性环节单位阶跃响应的解析解：y(t) = K(1 - e^{-t/tau})。'''
    return K * (1.0 - math.exp(-t / tau))

print(first_order_step(0.0))                 # 0.0
print(first_order_step(3.0))                 # 默认 K=1, tau=1 → 0.950
print(first_order_step(3.0, K=2.0, tau=0.5)) # 关键字传参，清晰不易错

# 多返回值（本质是返回一个元组）
def min_max(xs):
    '''返回序列的最小值和最大值。'''
    return min(xs), max(xs)

lo, hi = min_max([3, 1, 4, 1, 5, 9, 2, 6])   # 解包
print("min =", lo, ", max =", hi)

0.0
0.950212931632136
1.9950424956466672
min = 1 , max = 9


## 7. 作用域：变量在哪里生效

函数内部定义的变量是**局部变量**，函数结束即销毁；函数外的是**全局变量**。
函数内给一个变量赋值，默认创建同名局部变量，**不会**修改全局的那个。

工程建议：**别依赖全局变量传数据**——仿真参数通过函数参数显式传入，
否则代码长到几百行后，你根本追不到某个全局量在哪里被改掉了（这是调试地狱的入口）。

In [9]:
gain = 10          # 全局变量

def amp_bad(x):
    gain = 2       # 创建了同名【局部】变量，全局的 gain 不受影响
    return gain * x

def amp_good(x, gain=2):   # 推荐写法：参数显式传入
    return gain * x

print(amp_bad(1.0), "| 全局 gain 仍是:", gain)
print(amp_good(1.0, gain=10))

2.0 | 全局 gain 仍是: 10
10.0


## 8. 列表索引与切片入门

列表 `list` 是 Python 最常用的容器（下周 B02 深入）。索引从 **0** 开始，
负数从尾部数；切片 `xs[start:stop:step]` 取**左闭右开**区间——
对信号做「取一段、抽稀、反转」就是一行的事。

In [10]:
t = [round(0.1 * k, 1) for k in range(11)]       # 0.0 ~ 1.0 s 时间戳
sig = [round(first_order_step(ti, K=2.0, tau=0.5), 3) for ti in t]
print("时间:", t)
print("信号:", sig)

print("第一个样本:", sig[0], "| 最后一个:", sig[-1])
print("前 5 个:", sig[:5])          # start 省略 = 从头
print("第 3~7 个:", sig[3:8])       # 含 3 不含 8
print("隔一个取（抽稀）:", sig[::2])
print("反转:", sig[::-1])
print("切片长度:", len(sig[3:8]))   # 左闭右开：8 - 3 = 5

时间: [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]
信号: [0.0, 0.363, 0.659, 0.902, 1.101, 1.264, 1.398, 1.507, 1.596, 1.669, 1.729]
第一个样本: 0.0 | 最后一个: 1.729
前 5 个: [0.0, 0.363, 0.659, 0.902, 1.101]
第 3~7 个: [0.902, 1.101, 1.264, 1.398, 1.507]
隔一个取（抽稀）: [0.0, 0.659, 1.101, 1.398, 1.596, 1.729]
反转: [1.729, 1.669, 1.596, 1.507, 1.398, 1.264, 1.101, 0.902, 0.659, 0.363, 0.0]
切片长度: 5


## 小结与衔接

- 环境：`uv` + `.venv` + `uv.lock` = 隔离、可复现；运行代码一律 `uv run`；
- 语法：变量/数值、f-string、布尔 → `if` → `for/while` → 函数 → 作用域 → 列表切片；
- 你已经能用欧拉法仿真一阶环节了——B05 会把这条路走成专业的 `scipy.solve_ivp`。

**下周 B02**：把「一堆数」升级成「有组织的数据」——dict/tuple/推导式/文件读写/异常处理，
学会把仿真参数存成 JSON、把结果存成 CSV。

---

## ✏️ 练习

> 规则：先独立完成，再点开折叠的参考答案核对。交付物请直接写在本 notebook 末尾新建的 cell 中。

**练习 1（★，10 分钟，10 分）——生成正弦信号**
用 `for` 循环生成 $0 \le t < 1\,\mathrm{s}$、$T_s = 0.01\,\mathrm{s}$ 下正弦信号
$s(t) = \sin(2\pi \cdot 5 \cdot t)$（5 Hz）的所有样本，存入列表。
打印样本数、`max()` 和 `min()`。
**交付物**：代码 cell，输出样本数与最值（应分别为 100、约 1.0、约 -1.0）。

**练习 2（★，10 分钟，10 分）——把阻尼分类封装成函数**
把第 4 节的阻尼比分类逻辑封装成函数 `classify_zeta(zeta)`（返回描述字符串），
并用 `for` 循环对 `[-0.1, 0, 0.3, 1.0, 2.0]` 五个值逐个打印分类结果。
**交付物**：函数定义 + 循环测试 cell。

**练习 3（★★，20 分钟，20 分）——测一阶环节的上升时间**
上升时间 $t_r$ 定义为响应从稳态值 10% 上升到 90% 所需的时间。
用 `while`（或 `for`+`break`）循环对 $K=2,\ \tau=0.5,\ T_s=0.001$ 的欧拉仿真，
找出 $y$ 首次超过 $0.1K$ 和 $0.9K$ 的时刻，计算 $t_r$。
**交付物**：打印 $t_r$，并与理论值 $t_r \approx 2.2\tau$ 比较。

**练习 4（★，5 分钟，10 分）——概念与切片**
(a) 用两句话说明为什么项目要用 `uv` + 虚拟环境而不是全局 `pip install`；
(b) 给定 `sig = list(range(20))`，用切片取出「后半部分反转」的结果。
**交付物**：markdown 回答 (a) + 代码回答 (b)。

---

<details>
<summary>参考答案（做完再点开）</summary>

**练习 1**：

```python
import math
Ts = 0.01
sig = []
for k in range(100):
    t = k * Ts
    sig.append(math.sin(2 * math.pi * 5 * t))
print(len(sig), max(sig), min(sig))  # 100, ~0.9998, ~-0.9998
```

（5 Hz 在 1 s 内恰好 5 个整周期，但采样点不一定正好落在峰顶，所以最值略小于 1。）

**练习 2**：

```python
def classify_zeta(zeta):
    if zeta < 0:
        return "负阻尼（不稳定）"
    elif zeta == 0:
        return "无阻尼（等幅振荡）"
    elif zeta < 1:
        return "欠阻尼"
    elif zeta == 1:
        return "临界阻尼"
    else:
        return "过阻尼"

for z in [-0.1, 0, 0.3, 1.0, 2.0]:
    print(z, "->", classify_zeta(z))
```

**练习 3**：

```python
K, tau, Ts = 2.0, 0.5, 0.001
y, k = 0.0, 0
t10 = t90 = None
for k in range(10001):
    t = k * Ts
    if t10 is None and y >= 0.1 * K:
        t10 = t
    if t90 is None and y >= 0.9 * K:
        t90 = t
        break
    y += Ts * (K * 1.0 - y) / tau
print("t_r =", t90 - t10, "s;  2.2*tau =", 2.2 * tau)  # 应非常接近 1.1 s
```

**练习 4**：(a) 虚拟环境把不同项目的依赖相互隔离，避免版本冲突；
`uv.lock` 锁定确切版本，保证任何机器上 `uv sync` 后环境完全一致（可复现）。
(b) `sig[10:][::-1]` 或 `sig[:9:-1]`，结果为 `[19, 18, ..., 10]`。

</details>

---

## 延伸阅读

- [Python 官方教程（中文）：第 3~4 章（数据结构、控制流）](https://docs.python.org/zh-cn/3/tutorial/)
- [uv 官方文档](https://docs.astral.sh/uv/)
- [JupyterLab 文档](https://jupyterlab.readthedocs.io/)
- 前置阅读（选看）：《利用 Python 进行数据分析》第 1~3 章